# Module 1: Wrangle data

>[!NOTE]
> These links are not real. This the the claude-generated tutorial showing how public data can be wrangled from the internet.

In [ ]:
import urllib.request
import os

os.makedirs("data", exist_ok=True)

files = {
    "promoters.fa": "https://raw.githubusercontent.com/kieranrcampbell/genomics-ml-tutorial/main/data/promoters.fa",
    "expression.tsv": "https://raw.githubusercontent.com/kieranrcampbell/genomics-ml-tutorial/main/data/expression.tsv"
}

for filename, url in files.items():
    path = f"data/{filename}"
    if not os.path.exists(path):
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, path)
        print(f"  Saved to {path}")
    else:
        print(f"  {filename} already exists, skipping.")

### Generate some synthetic training data

In [3]:
import numpy as np
import pandas as pd
from Bio.Seq import Seq
import os

os.makedirs("data", exist_ok=True)
rng = np.random.default_rng(seed=42)

# --- Biological parameters ---
N_GENES = 1000
PROMOTER_LENGTH = 200  # shortened for speed; real promoters ~1000bp

# Two real yeast transcription factor binding motifs
# TATA box (associated with high expression in many genes)
TATA_BOX = "TATAAA"
# Poly-A stretch (associated with nucleosome positioning)
POLYA = "AAAAAA"

BASES = ['A', 'T', 'G', 'C']

def random_sequence(length, rng, gc_content=0.38):
    """Generate a random DNA sequence with specified GC content."""
    at = (1 - gc_content) / 2
    gc = gc_content / 2
    probs = [at, at, gc, gc]  # A, T, G, C
    return ''.join(rng.choice(BASES, size=length, p=probs))

def insert_motif(seq, motif, position):
    """Insert a motif at a given position in a sequence string."""
    seq = list(seq)
    for i, base in enumerate(motif):
        seq[position + i] = base
    return ''.join(seq)

sequences = []
labels = []
metadata = []

for i in range(N_GENES):
    seq = random_sequence(PROMOTER_LENGTH, rng)
    
    # High-expression genes get a TATA box planted near position 30
    # plus slightly higher AT content (real biology)
    if i < N_GENES // 2:
        label = 1
        seq = insert_motif(seq, TATA_BOX, position=30)
        # Sometimes also plant a secondary motif (adds noise)
        if rng.random() < 0.4:
            seq = insert_motif(seq, POLYA, position=100)
    else:
        label = 0
        # Occasionally a low-expression gene has a TATA box anyway (noise)
        if rng.random() < 0.1:
            seq = insert_motif(seq, TATA_BOX, position=30)

    sequences.append(seq)
    labels.append(label)
    metadata.append(f"gene_{i:04d}")

# Save as FASTA
fasta_path = "data/promoters.fa"
with open(fasta_path, "w") as f:
    for name, seq in zip(metadata, sequences):
        f.write(f">{name}\n{seq}\n")

# Save labels
labels_df = pd.DataFrame({"gene_id": metadata, "high_expression": labels})
labels_df.to_csv("data/expression.tsv", sep="\t", index=False)

print(f"Generated {N_GENES} promoter sequences of length {PROMOTER_LENGTH}bp")
print(f"High expression: {sum(labels)}, Low expression: {N_GENES - sum(labels)}")
print(f"Files saved to data/")

Generated 1000 promoter sequences of length 200bp
High expression: 500, Low expression: 500
Files saved to data/


### Load and inspect the data

In [4]:
from Bio import SeqIO

records = list(SeqIO.parse("data/promoters.fa", "fasta"))
labels_df = pd.read_csv("data/expression.tsv", sep="\t")

print(f"Number of sequences: {len(records)}")
print(f"\nExample record:")
print(f"  ID: {records[0].id}")
print(f"  Sequence: {str(records[0].seq)[:50]}...")
print(f"  Length: {len(records[0].seq)}bp")
print(f"\nLabels dataframe:")
print(labels_df.head())
print(f"\nLabel distribution:\n{labels_df['high_expression'].value_counts()}")

Number of sequences: 1000

Example record:
  ID: gene_0000
  Sequence: GTCGACGGATTCGCTATACGGTCCGATAAGTATAAAATAGTCGTCGTAGA...
  Length: 200bp

Labels dataframe:
     gene_id  high_expression
0  gene_0000                1
1  gene_0001                1
2  gene_0002                1
3  gene_0003                1
4  gene_0004                1

Label distribution:
high_expression
1    500
0    500
Name: count, dtype: int64
